In [25]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from bonXAI.core.preprocessor import Preprocessor
from bonXAI.core.explainer import Explainer
from bonXAI.core.evaluation import Evaluator
from bonXAI.core.utils import set_global_seed, possible_g_values, possible_num_bins_values

from openxai.model import LoadModel, ReturnLoaders

# pip install stein-thinning https://zoltansz.github.io/utils/DSS/slides/2022_05_30_Lester_Mackey_slides.pdf http://stein-thinning.org
from stein_thinning.thinning import thin

SEED = 42
set_global_seed(SEED)

In [26]:
# generate data
X, y = make_classification(n_samples=1000, n_features=7, n_classes=2, random_state=SEED)
df_data = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df_data["label"] = y

In [27]:
# train model
model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=SEED)
model.fit(X, y)

MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=42)

In [ ]:
# calculate Ground Truth
gt_explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
exp_gt, t_gt = gt_explainer.explain(X, y)
exp_gt_mean = exp_gt.mean(axis=0)

Using 1000 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


### Kernel thinning

### Stein Thinnings KST

In [28]:
# Fit a Gaussian to X and compute score (grad log p)
mu = X.mean(axis=0)
Sigma = np.cov(X, rowvar=False) + 1e-6*np.eye(X.shape[1])  # tiny ridge for stability
prec = np.linalg.inv(Sigma)
grad = -(X - mu) @ prec     # shape: (n, d)

In [29]:
m = 16  # how many points to keep
idx = thin(X, grad, m)      # indices of selected rows  :contentReference[oaicite:1]{index=1}

X_sel = X[idx]
y_sel = y[idx]

In [30]:
X_sel.shape

(16, 7)

### UPDATE TO STEIN THINNING - methods for grad

In [31]:
import sys
import os
sys.path.append("..")
import time
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from bonXAI.core.bonxai_compress import bonxai_compress
from bonXAI.core.preprocessor import Preprocessor, Compressor
from bonXAI.core.metrics import compute_mmd 
from openxai.dataloader import ReturnLoaders

import numpy as np
from sklearn.metrics.pairwise import rbf_kernel

_, loader_test = ReturnLoaders(data_name="heloc", download=False, batch_size=128)
X = loader_test.dataset.data
y = loader_test.dataset.targets.to_numpy()

In [32]:
import time
from typing import Optional, Tuple
import numpy as np
from sklearn.mixture import GaussianMixture

def stein_thinning(
    X, m: int, grad_type: bytes, n_components: int = 2
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    grad = None    
    start = time.time()

    if grad_type == b'gaussian':
        mu = X.mean(axis=0)
        Sigma = np.cov(X, rowvar=False) + 1e-6*np.eye(X.shape[1])  
        prec = np.linalg.inv(Sigma)
        grad = -(X - mu) @ prec 
    
    elif grad_type == b'kde':
        n, d = X.shape
        std_dev = X.std(axis=0, ddof=1)
        bandwidth = np.mean(std_dev) * (4 / (d + 2) / n) ** (1 / (d + 4))
        diffs = X[:, None, :] - X[None, :, :]  # shape (n, n, d)
        sq_dist = np.sum(diffs**2, axis=2)     # shape (n, n)
        weights = np.exp(-0.5 * sq_dist / bandwidth**2)  # shape (n, n)
        grad = -np.einsum('ijk,ij->ik', diffs, weights) / (bandwidth**2 * n)

    elif grad_type == b'gmm':
        n, d = X.shape
        gmm = GaussianMixture(n_components=n_components, covariance_type='full')
        gmm.fit(X)

        mu = gmm.means_           # (K, d)
        cov = gmm.covariances_    # (K, d, d)
        prec = np.linalg.inv(cov) # (K, d, d)
        resp = gmm.predict_proba(X)  # (n, K)

        grad = np.zeros((n, d))
        for k in range(n_components):
            diff = mu[k] - X              # (n, d)
            grad += resp[:, [k]] * (diff @ prec[k])  # (n, d)

    if grad is None:
        raise ValueError("Gradient must be provided for non-Gaussian Stein thinning.")
    
    indices = thin(X, grad, m)  
    end = time.time()
    return X[indices]


In [33]:
X1 = stein_thinning(X, m=16, grad_type=b'kde')

In [34]:
X2 = stein_thinning(X, m=16, grad_type=b'gaussian')

In [35]:
X3 = stein_thinning(X, m=16, grad_type=b'gmm')

In [36]:
pre = Preprocessor(X=X, y=y, model=None, compression_method="kernel_thinning")
X_kt, y_kt, idx_kt, t_kt = pre._preprocess(kernel="inverse_multiquadric")
X_kt.shape

(32, 23)

In [37]:
compute_mmd(X, X1)

0.031494981732033134

In [38]:
compute_mmd(X, X2)

0.015916533097194208

In [39]:
compute_mmd(X, X3)

0.005735344096870376

In [40]:
compute_mmd(X_kt, X)

0.000432761744027399

In [41]:
X_kt.shape

(32, 23)

In [42]:
from goodpoints.compress import compresspp_kt

X_reduced = sorted(compresspp_kt(X=X, kernel_type=b"sobolev"))
compute_mmd(X, X[X_reduced])

0.0029802240067888786

In [43]:
X_reduced = sorted(compresspp_kt(X=X, kernel_type=b"gaussian"))
compute_mmd(X, X[X_reduced])

0.0004589209174203912

In [44]:
X_reduced = sorted(compresspp_kt(X=X, kernel_type=b"inverse_multiquadric"))
compute_mmd(X, X[X_reduced])

0.0004157842596035355

In [49]:
import numpy as np

# Flattened 1D params for a single Matérn kernel: [sigma, ell, nu]
k_params = np.array([ 1.0, 1.0, 1.5], dtype=np.float64) # we can use either nu=0.5, 1.5, or 2.5

X_reduced = compresspp_kt(X=X, kernel_type=b"matern", k_params=k_params)
compute_mmd(X, X[X_reduced])

0.00037563218725278347